# Data Modeling

Prepare an analysis-ready dataset by integrating the cleaned trading and market sentiment data. The resulting dataset will serve as the foundation for subsequent analysis and insight generation.

## Import Libraries

Import the required libraries for data modeling.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

## Load Cleaned Datasets

Load the cleaned datasets generated during the data cleaning phase.

In [2]:
historical_df = pd.read_csv(
    "../data/processed/historical_data_clean.csv",
    parse_dates=["Timestamp IST", "Timestamp"]
)

sentiment_df = pd.read_csv(
    "../data/processed/fear_greed_index_clean.csv",
    parse_dates=["date", "timestamp"]
)

## Create Working Copies

Create working copies of the cleaned datasets to preserve the original data throughout the modeling process.

In [3]:
historical_model = historical_df.copy()
sentiment_model = sentiment_df.copy()

## Create Analysis Date

Create a common date field to support daily-level integration between trader activity and market sentiment.

In [4]:
historical_model["Date"] = historical_model["Timestamp IST"].dt.normalize()

sentiment_model["Date"] = sentiment_model["date"].dt.normalize()

## Date Coverage

Review the date range of each dataset to validate the overlap available for integration.

In [5]:
date_coverage = pd.DataFrame({
    "Dataset": ["Historical Data", "Fear & Greed Data"],
    "Start Date": [
        historical_model["Date"].min(),
        sentiment_model["Date"].min()
    ],
    "End Date": [
        historical_model["Date"].max(),
        sentiment_model["Date"].max()
    ]
})

display(date_coverage)

,Dataset,Start Date,End Date
0,Historical Data,2023-05-01,2025-05-01
1,Fear & Greed Data,2018-02-01,2025-05-02


## Integrate Datasets

Combine the trading and market sentiment datasets using the common analysis date to create a unified analytical model.

In [6]:
analysis_df = historical_model.merge(
    sentiment_model[["Date", "classification", "value"]],
    on="Date",
    how="left"
)

In [7]:
analysis_summary = pd.DataFrame({
    "Rows": [analysis_df.shape[0]],
    "Columns": [analysis_df.shape[1]],
    "Unmatched Sentiment Records": [analysis_df["classification"].isna().sum()]
})

display(analysis_summary)

,Rows,Columns,Unmatched Sentiment Records
0,211224,19,6


## Handle Missing Sentiment

Populate missing market sentiment values using the most recent available observation to preserve complete trader activity during integration.

In [8]:
analysis_df = analysis_df.sort_values("Date")

analysis_df[["classification", "value"]] = (
    analysis_df[["classification", "value"]]
    .ffill()
)

## Validate Integration

Verify that all trading records have been successfully aligned with market sentiment.

In [9]:
integration_summary = pd.DataFrame({
    "Rows": [analysis_df.shape[0]],
    "Columns": [analysis_df.shape[1]],
    "Unmatched Sentiment Records": [
        analysis_df["classification"].isna().sum()
    ]
})

display(integration_summary)

,Rows,Columns,Unmatched Sentiment Records
0,211224,19,0


## Create Date Features

Generate calendar attributes to support time-based analysis and reporting.

In [10]:
analysis_df["Year"] = analysis_df["Date"].dt.year
analysis_df["Quarter"] = analysis_df["Date"].dt.quarter
analysis_df["Month"] = analysis_df["Date"].dt.month
analysis_df["Month Name"] = analysis_df["Date"].dt.month_name()
analysis_df["Week"] = analysis_df["Date"].dt.isocalendar().week.astype(int)
analysis_df["Day"] = analysis_df["Date"].dt.day
analysis_df["Day Name"] = analysis_df["Date"].dt.day_name()

## Organize Dataset

Arrange the integrated dataset in chronological order to maintain a consistent record sequence.

In [11]:
analysis_df = (
    analysis_df
    .sort_values(
        by=["Date", "Timestamp IST", "Order ID"],
        ascending=True
    )
    .reset_index(drop=True)
)

## Validate Date Features

Review the newly created calendar attributes.

In [12]:
display(
    analysis_df[
        [
            "Date",
            "Year",
            "Quarter",
            "Month",
            "Month Name",
            "Week",
            "Day",
            "Day Name"
        ]
    ].head()
)

,Date,Year,Quarter,Month,Month Name,Week,Day,Day Name
0,2023-05-01,2023,2,5,May,18,1,Monday
1,2023-05-01,2023,2,5,May,18,1,Monday
2,2023-05-01,2023,2,5,May,18,1,Monday
3,2023-12-05,2023,4,12,December,49,5,Tuesday
4,2023-12-05,2023,4,12,December,49,5,Tuesday


## Export Modeled Dataset

Save the integrated dataset for use in subsequent analysis notebooks.

In [13]:
analysis_df.to_csv(
    "../data/processed/trader_sentiment_model.csv",
    index=False
)

print("Modeled dataset saved successfully.")

Modeled dataset saved successfully.


## Summary

The datasets were successfully integrated into a unified analytical model.

- Combined trader activity with market sentiment using a common date.
- Addressed missing sentiment observations through forward filling.
- Generated reusable calendar features.
- Organized records in chronological order.
- Exported the analysis-ready dataset for subsequent project stages.